In [2]:
import torch
from torch import nn
import os
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

In [3]:
!unzip archive.zip

Streaming output truncated to the last 5000 lines.
  inflating: processed_data/surprise/surprise_00920.jpg  
  inflating: processed_data/surprise/surprise_00921.jpg  
  inflating: processed_data/surprise/surprise_00922.jpg  
  inflating: processed_data/surprise/surprise_00923.jpg  
  inflating: processed_data/surprise/surprise_00924.jpg  
  inflating: processed_data/surprise/surprise_00925.jpg  
  inflating: processed_data/surprise/surprise_00926.jpg  
  inflating: processed_data/surprise/surprise_00927.jpg  
  inflating: processed_data/surprise/surprise_00928.jpg  
  inflating: processed_data/surprise/surprise_00929.jpg  
  inflating: processed_data/surprise/surprise_00930.jpg  
  inflating: processed_data/surprise/surprise_00931.jpg  
  inflating: processed_data/surprise/surprise_00932.jpg  
  inflating: processed_data/surprise/surprise_00933.jpg  
  inflating: processed_data/surprise/surprise_00934.jpg  
  inflating: processed_data/surprise/surprise_00935.jpg  
  inflating: processe

In [1]:
!pip install split-folders

In [4]:
import splitfolders
splitfolders.ratio(input="processed_data",output="dataset_split",seed=42,ratio=(.8,.0,.2))

Copying files: 49779 files [00:07, 6867.91 files/s]


In [5]:
from pathlib import Path
data_path = Path("")
image_path = data_path / "dataset_split"
train_path = image_path / "train"
test_path = image_path / "test"

In [6]:
def check_data(dir_path):
    for dirpath,dirnames,filenames in os.walk(dir_path):
        print(f"# of directories in '{len(dirnames)}' and {len(filenames)} images in {dirpath}")



In [7]:
check_data(image_path)

# of directories in '3' and 0 images in dataset_split
# of directories in '7' and 0 images in dataset_split/test
# of directories in '0' and 1307 images in dataset_split/test/sad
# of directories in '0' and 1184 images in dataset_split/test/angry
# of directories in '0' and 1634 images in dataset_split/test/neutral
# of directories in '0' and 1184 images in dataset_split/test/surprise
# of directories in '0' and 2280 images in dataset_split/test/happy
# of directories in '0' and 1184 images in dataset_split/test/fear
# of directories in '0' and 1184 images in dataset_split/test/disgust
# of directories in '7' and 0 images in dataset_split/train
# of directories in '0' and 5228 images in dataset_split/train/sad
# of directories in '0' and 4736 images in dataset_split/train/angry
# of directories in '0' and 6532 images in dataset_split/train/neutral
# of directories in '0' and 4736 images in dataset_split/train/surprise
# of directories in '0' and 9118 images in dataset_split/train/happy

In [8]:
NUM_WORKERS = os.cpu_count()

def create_dataloader(train_dir,
                      test_dir,
                      transforms: transforms.Compose,
                      batch_size:int,
                      workers:int = NUM_WORKERS):

    train_data = datasets.ImageFolder(root=train_dir,
                                      transform=transforms)
    test_data = datasets.ImageFolder(root=test_dir,
                                     transform=transforms)

    class_names = train_data.classes

    train_dataloader = DataLoader(dataset=train_data,
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=workers)

    test_dataloader = DataLoader(dataset=test_data,
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=workers)
    return train_dataloader, test_dataloader, class_names

In [57]:
weight = models.EfficientNet_B7_Weights.DEFAULT

In [58]:
auto_transforms = weight.transforms()

In [59]:
auto_transforms

ImageClassification(
    crop_size=[600]
    resize_size=[600]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [75]:
train_dataloader, test_dataloader, class_names = create_dataloader(train_dir=train_path,
                                                                   test_dir=test_path,
                                                                   transforms=auto_transforms,
                                                                   batch_size=32,)

In [76]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [77]:
model = models.efficientnet_b7(weights=weight).to(device)

In [17]:
!pip install torchinfo

In [63]:
from torchinfo import summary
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 1000]                --                        True
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 2560, 3, 3]          --                        True
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 64, 48, 48]          --                        True
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 64, 48, 48]          1,728                     True
│    │    └─BatchNorm2d: 3-2                            [32, 64, 48, 48]          [32, 64, 48, 48]          128                       True
│    │    └─SiLU: 3-3                                   [32, 64, 48, 48]          [32, 64, 48, 48]          --                        --
│    └─Sequential: 2-2  

In [64]:
#for params in model.parameters():
    #params.requires_grad = False

In [65]:
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 1000]                --                        True
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 2560, 3, 3]          --                        True
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 64, 48, 48]          --                        True
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 64, 48, 48]          1,728                     True
│    │    └─BatchNorm2d: 3-2                            [32, 64, 48, 48]          [32, 64, 48, 48]          128                       True
│    │    └─SiLU: 3-3                                   [32, 64, 48, 48]          [32, 64, 48, 48]          --                        --
│    └─Sequential: 2-2  

In [66]:
output_shape = len(class_names)
output_shape

7

In [67]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
            (1): BatchNorm2d(64, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormAct

In [68]:
model.fc = nn.Sequential(
    torch.nn.Linear(in_features=2560, out_features=output_shape)
)

In [69]:
#for params in model.classifier.parameters():
    #params.requires_grad = True

In [70]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
            (1): BatchNorm2d(64, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormAct

In [71]:
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 1000]                17,927                    True
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 2560, 3, 3]          --                        True
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 64, 48, 48]          --                        True
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 64, 48, 48]          1,728                     True
│    │    └─BatchNorm2d: 3-2                            [32, 64, 48, 48]          [32, 64, 48, 48]          128                       True
│    │    └─SiLU: 3-3                                   [32, 64, 48, 48]          [32, 64, 48, 48]          --                        --
│    └─Sequential: 2-2  

In [78]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(),lr = 0.0003)

In [79]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device
              ):
    model.train() # Modelimizi train moduna alıyoruz.

    train_loss = 0 # Train Loss değerlerini tutmak için bir değişken oluşturuyoruz.
    train_acc = 0 # Train Accuracy değerlerini tutmak için bir değişken oluşturuyoruz.

    for batch, (X,y) in enumerate(dataloader): # Batch size gerekli değil burada.
        X,y = X.to(device), y.to(device)
        y_pred = model(X) # Modelimize bir tahminde bulunduruyoruz.

        loss = loss_fn(y_pred,y) # Loss değerlerimizi loss_fn ile hesaplıyoruz.
        train_loss += loss.item() # Çıkan loss değerlerini train_loss değişlenine toplayarak atıyoruz.

        # Modelimizi backpropagation yapıyoruz.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Softmax kullanarak modelimize label tahmininde bulunduruyoruz.
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

        train_acc += (y_pred_class == y).sum().item() / len(y_pred) # Accuracy değerlerimizi bir değişkende tutuyoruz.

    train_loss /= len(dataloader) # Train Loss değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz.
    train_acc /= len(dataloader) # Train Acc değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz
    return train_loss, train_acc # Geriye train_loss ve train_acc döndürüyoruz.

def test_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
              device: torch.device
              ):
    model.eval() # Modelimizi test moduna alıyoruz.

    test_loss = 0 # test loss'ları tutmak için bir değişken oluşturuyoruz.
    test_acc = 0 # test accuracy'ları tutmak için bir değişken oluşturuyoruz.

    with torch.inference_mode(): # inference mode'a aldık.
        for batch, (X,y) in enumerate(dataloader): # batch gerekli değil fakat yine de aldık.
            X,y = X.to(device), y.to(device)
            test_pred = model(X) # modelimize tahmin ettiriyoruz.

            loss = loss_fn(test_pred,y) # loss'umuzu loss_fn ile hesaplıyoruz.
            test_loss += loss.item() # loss değerlerimizi test_loss değişkeninde topluyoruz.

            # Softmax activation function ile label tahmin ettiriyoruz.
            test_pred_label = torch.softmax(test_pred,dim=1).argmax(dim=1)

            acc = (test_pred_label == y).sum().item() / len(test_pred) # Calculate accuracy
            test_acc += acc # Accuracy değerlerimizi toplayıp test_acc değişkenine atıyoruz.

    test_loss /= len(dataloader) # Test loss değerlerimizi dataloader boyuna bölüyoruz.
    test_acc /= len(dataloader) # Test acc değerlerimizi dataloader boyuna bölüyoruz.

    return test_loss, test_acc # Geriye test_loss ve test_acc döndürüyoruz.

def train(model: torch.nn.Module,
               train_dataloader: torch.utils.data.DataLoader,
               test_dataloader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               device:torch.device,
               loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
               epochs:int = 10,
              ):
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer,
                                           device = device
                                          )
        test_loss, test_acc = test_step(model = model,
                                           dataloader = test_dataloader,
                                           loss_fn = loss_fn,
                                           device = device
                                          )
        print(f"""
        Epoch:{epoch}
        Train Loss : {train_loss:.2f} -  Train Accuracy : {train_acc*100:.2f}
        Test Loss  : { test_loss:.2f} -  Test Accuracy  : {test_acc*100:.2f}
        """)
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)
    return results

In [80]:
results = train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=torch.device("cuda"),
                epochs=10)

OutOfMemoryError: CUDA out of memory. Tried to allocate 704.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 143.81 MiB is free. Including non-PyTorch memory, this process has 14.42 GiB memory in use. Of the allocated memory 14.27 GiB is allocated by PyTorch, and 12.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)